# Apache Polaris — Clean All Resources





## ⚠️ Warning
This notebook deletes ALL Polaris resources except system entities (root, service_admin).

**Deletion order matters** — must follow dependency order:
```
1. Tables & Views      (inside namespaces)
2. Namespaces          (DEEPEST-FIRST — child namespaces before parents)
3. Catalog Roles       (non-default; a catalog with extra roles is "not empty")
4. Catalogs
5. Principals          (except root)
6. Principal Roles     (except service_admin)
```

### Nested namespaces
Polaris namespaces can be multi-level (e.g. `parent.child`). The REST API joins
the levels with the ASCII Unit Separator `\x1f` (shown as `%1F` in URLs). This
notebook lists namespaces recursively and deletes them deepest-first, so a parent
namespace is never deleted before its children. A namespace must be EMPTY
(no tables/views/child-namespaces) before it can be dropped.


In [45]:
# ============================================================
# Prerequisites:  uv add requests pandas
# ============================================================
import requests, json
import pandas as pd
from IPython.display import display

POLARIS_URL = 'http://192.168.139.2:8181'

# ── SAFETY GUARD: LOCAL DEVICE ONLY ─────────────────────────────────────────
# This notebook is fully destructive (wipes ALL catalogs/principals/roles).
# It must never touch a shared company environment. Hard-fail on non-local hosts.
assert any(t in POLARIS_URL for t in ("127.0.0.1", "localhost", "192.168.139.2")), \
    f"clean_all is LOCAL ONLY — refusing non-local host: {POLARIS_URL}"
REALM       = 'POLARIS'
NS_SEP      = '\x1f'   # Polaris namespace separator (shows as %1F in URLs)

def get_token(client_id='root', client_secret='polaris-secret'):
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},
        data={'grant_type': 'client_credentials', 'client_id': client_id,
              'client_secret': client_secret, 'scope': 'PRINCIPAL_ROLE:ALL'})
    return r.json()['access_token']

def h(token):
    return {'Authorization': f'Bearer {token}', 'Polaris-Realm': REALM,
            'Content-Type': 'application/json'}

def delete(url, token, label):
    r = requests.delete(url, headers=h(token))
    icon = '✅' if r.status_code in (200, 204) else '⚠️ '
    print(f'  {icon} [{r.status_code}] {label}')
    return r.status_code

# ── Namespace helpers (handle multi-level namespaces) ──
def ns_parts(ns):
    """Normalize a namespace (list, or %1F/\x1f-joined string) to a list of parts."""
    if isinstance(ns, list):
        return ns
    return ns.replace('%1F', NS_SEP).split(NS_SEP)

def ns_path(ns):
    """Namespace → URL path segment using the raw separator (requests encodes it)."""
    return NS_SEP.join(ns_parts(ns))

def ns_label(ns):
    """Human-readable dotted namespace name."""
    return '.'.join(ns_parts(ns))

def list_namespaces(cname, token):
    """Return ALL namespaces in a catalog (Polaris list is already recursive)."""
    r = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces', headers=h(token))
    return r.json().get('namespaces', [])

def namespaces_deep_first(cname, token):
    """Namespaces sorted DEEPEST-first so children are deleted before parents."""
    return sorted(list_namespaces(cname, token), key=lambda n: len(ns_parts(n)), reverse=True)

TOKEN = get_token()
SKIP_PRINCIPALS      = {'root'}
SKIP_PRINCIPAL_ROLES = {'service_admin'}

print('✅ Connected as root')
print(f'   Skip principals:      {SKIP_PRINCIPALS}')
print(f'   Skip principal roles: {SKIP_PRINCIPAL_ROLES}')


✅ Connected as root
   Skip principals:      {'root'}
   Skip principal roles: {'service_admin'}


In [46]:
# ============================================================
# Preview — show what will be deleted (recursive)
# ============================================================
print('=== Resources to be deleted ===\n')

catalogs        = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs',        headers=h(TOKEN)).json().get('catalogs', [])
principals      = requests.get(f'{POLARIS_URL}/api/management/v1/principals',      headers=h(TOKEN)).json().get('principals', [])
principal_roles = requests.get(f'{POLARIS_URL}/api/management/v1/principal-roles', headers=h(TOKEN)).json().get('roles', [])

print(f'Catalogs ({len(catalogs)}):')
for c in catalogs:
    cname = c['name']
    print(f'  📚 {cname}')
    # catalog roles
    for cr in requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN)).json().get('roles', []):
        tag = ' (system)' if cr['name'] == 'catalog_admin' else ''
        print(f'    🔑 catalog-role: {cr["name"]}{tag}')
    # namespaces (deepest-first, with contents)
    for ns in namespaces_deep_first(cname, TOKEN):
        path = ns_path(ns)
        tbls = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}/tables', headers=h(TOKEN)).json().get('identifiers', [])
        vws  = requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}/views',  headers=h(TOKEN)).json().get('identifiers', [])
        depth = len(ns_parts(ns))
        print(f'    📁 namespace: {ns_label(ns)}  (depth={depth}, tables={len(tbls)}, views={len(vws)})')
        for t in tbls: print(f'        📋 table: {t.get("name")}')
        for v in vws:  print(f'        👁️  view:  {v.get("name")}')

print(f'\nPrincipals ({len(principals)}):')
for p in principals:
    skip = ' (skip)' if p['name'] in SKIP_PRINCIPALS else ''
    print(f'  👤 {p["name"]}{skip}')
print(f'Principal Roles ({len(principal_roles)}):')
for pr in principal_roles:
    skip = ' (skip)' if pr['name'] in SKIP_PRINCIPAL_ROLES else ''
    print(f'  🎭 {pr["name"]}{skip}')


=== Resources to be deleted ===

Catalogs (0):

Principals (1):
  👤 root (skip)
Principal Roles (1):
  🎭 service_admin (skip)


In [36]:
# ============================================================
# Step 1 — Delete Tables and Views (all namespaces, recursive)
# ============================================================
print('=== Step 1: Delete Tables and Views ===')

for c in catalogs:
    cname = c['name']
    for ns in namespaces_deep_first(cname, TOKEN):
        path = ns_path(ns); lbl = ns_label(ns)
        # tables
        for t in requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}/tables', headers=h(TOKEN)).json().get('identifiers', []):
            tname = t.get('name', str(t))
            delete(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}/tables/{tname}',
                   TOKEN, f'table: {cname}.{lbl}.{tname}')
        # views
        for v in requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}/views', headers=h(TOKEN)).json().get('identifiers', []):
            vname = v.get('name', str(v))
            delete(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}/views/{vname}',
                   TOKEN, f'view:  {cname}.{lbl}.{vname}')

print('\n✅ Step 1 complete')


=== Step 1: Delete Tables and Views ===

✅ Step 1 complete


In [37]:
# ============================================================
# Step 1.5 — Enable drop-with-purge on every catalog (so drops aren't blocked)
# ============================================================
# During teardown we want purge ENABLED so a view/table drop can't be blocked by
# the config gate. Set it with a verified, retry-on-409 PUT.
print('=== Step 1.5: Enable drop-with-purge.enabled=true on all catalogs ===')

def set_catalog_prop(cname, key, value, retries=4):
    url = f'{POLARIS_URL}/api/management/v1/catalogs/{cname}'
    for _ in range(retries):
        r = requests.get(url, headers=h(TOKEN))
        if r.status_code != 200:
            print(f'  ⚠️  {cname}: GET {r.status_code}'); return False
        cat = r.json()
        # Polaris returns the catalog either at top-level or under "catalog"
        cat_obj = cat.get('catalog', cat)
        version = cat_obj.get('entityVersion', cat.get('entityVersion', 1))
        props   = dict(cat_obj.get('properties', {}) or {})
        if props.get(key) == value:
            return True
        props[key] = value
        put = requests.put(url, headers=h(TOKEN), json={
            'currentEntityVersion': version, 'properties': props})
        if put.status_code in (200, 204):
            chk = requests.get(url, headers=h(TOKEN)).json()
            chk_obj = chk.get('catalog', chk)
            if (chk_obj.get('properties', {}) or {}).get(key) == value:
                print(f'  ✅ {cname}: {key}={value}')
                return True
        elif put.status_code == 409:
            continue   # version conflict → re-read and retry
        else:
            print(f'  ⚠️  {cname}: PUT {put.status_code} {put.text[:80]}')
            return False
    print(f'  ⚠️  {cname}: could not set {key}')
    return False

for c in catalogs:
    set_catalog_prop(c['name'], 'polaris.config.drop-with-purge.enabled', 'true')

print('\n✅ Step 1.5 complete')


=== Step 1.5: Enable drop-with-purge.enabled=true on all catalogs ===

✅ Step 1.5 complete


In [38]:
# (Step 1.5 above consolidated the catalog-property handling. This cell intentionally
#  left as a no-op to preserve notebook cell numbering.)
print('→ catalog properties handled in Step 1.5')


→ catalog properties handled in Step 1.5


In [39]:
# ============================================================
# Step 2 — Delete Namespaces (DEEPEST-FIRST, recursive-safe)
# ============================================================
print('=== Step 2: Delete Namespaces (deepest-first) ===')

for c in catalogs:
    cname = c['name']
    # Re-list each pass; deleting children can make parents deletable.
    # Loop until no namespaces remain or no progress is made.
    for _pass in range(6):
        nss = namespaces_deep_first(cname, TOKEN)
        if not nss:
            break
        progressed = False
        for ns in nss:
            path = ns_path(ns); lbl = ns_label(ns)
            # ensure empty: drop any lingering tables/views first
            for t in requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}/tables', headers=h(TOKEN)).json().get('identifiers', []):
                delete(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}/tables/{t["name"]}', TOKEN, f'  (leftover) table {lbl}.{t["name"]}')
            for v in requests.get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}/views', headers=h(TOKEN)).json().get('identifiers', []):
                delete(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}/views/{v["name"]}', TOKEN, f'  (leftover) view {lbl}.{v["name"]}')
            code = delete(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{path}',
                          TOKEN, f'namespace: {cname}.{lbl}')
            if code in (200, 204):
                progressed = True
        if not progressed:
            print(f'  ⚠️  {cname}: no namespace progress this pass — remaining: {[ns_label(n) for n in namespaces_deep_first(cname, TOKEN)]}')
            break

print('\n✅ Step 2 complete')


=== Step 2: Delete Namespaces (deepest-first) ===
  ✅ [204] namespace: user10_catalog.ns1
  ✅ [204] namespace: user10_catalog.ns2
  ✅ [204] namespace: user16_catalog.ns1
  ✅ [204] namespace: user16_catalog.ns2
  ✅ [204] namespace: user19_catalog.ns1
  ✅ [204] namespace: user19_catalog.ns2
  ✅ [204] namespace: user1_catalog.ns1
  ✅ [204] namespace: user1_catalog.ns2
  ✅ [204] namespace: user20_catalog.ns1
  ✅ [204] namespace: user20_catalog.ns2
  ✅ [204] namespace: user22_catalog.ns1
  ✅ [204] namespace: user22_catalog.ns2
  ✅ [204] namespace: user25_catalog.ns1
  ✅ [204] namespace: user25_catalog.ns2
  ✅ [204] namespace: user26_catalog.ns1
  ✅ [204] namespace: user26_catalog.ns2
  ✅ [204] namespace: user28_catalog.ns1
  ✅ [204] namespace: user28_catalog.ns2
  ✅ [204] namespace: user29_catalog.ns1
  ✅ [204] namespace: user29_catalog.ns2
  ✅ [204] namespace: user2_catalog.ns1
  ✅ [204] namespace: user2_catalog.ns2
  ✅ [204] namespace: user30_catalog.ns1
  ✅ [204] namespace: user30_catalo

In [40]:
# ============================================================
# Step 3 — Delete Catalog Roles
# ============================================================
print('=== Step 3: Delete Catalog Roles ===')

for c in catalogs:
    cname = c['name']
    cr_r = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN))
    for cr in cr_r.json().get('roles', []):
        crname = cr['name']
        if crname == 'catalog_admin':
            print(f'  ⏭️  skip: {cname}/{crname} (system role)')
            continue
        delete(
            f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles/{crname}',
            TOKEN, f'catalog-role: {cname}/{crname}'
        )

print('\n✅ Step 3 complete')

=== Step 3: Delete Catalog Roles ===
  ⏭️  skip: user10_catalog/catalog_admin (system role)
  ✅ [204] catalog-role: user10_catalog/owner_principal
  ⏭️  skip: user16_catalog/catalog_admin (system role)
  ✅ [204] catalog-role: user16_catalog/owner_principal
  ⏭️  skip: user19_catalog/catalog_admin (system role)
  ✅ [204] catalog-role: user19_catalog/owner_principal
  ⏭️  skip: user1_catalog/catalog_admin (system role)
  ✅ [204] catalog-role: user1_catalog/owner_principal
  ⏭️  skip: user20_catalog/catalog_admin (system role)
  ✅ [204] catalog-role: user20_catalog/owner_principal
  ⏭️  skip: user22_catalog/catalog_admin (system role)
  ✅ [204] catalog-role: user22_catalog/owner_principal
  ⏭️  skip: user25_catalog/catalog_admin (system role)
  ✅ [204] catalog-role: user25_catalog/owner_principal
  ⏭️  skip: user26_catalog/catalog_admin (system role)
  ✅ [204] catalog-role: user26_catalog/owner_principal
  ⏭️  skip: user28_catalog/catalog_admin (system role)
  ✅ [204] catalog-role: user28

In [41]:
# ============================================================
# Step 4 — Delete Catalogs (with verification)
# ============================================================
print('=== Step 4: Delete Catalogs ===')

for c in catalogs:
    cname = c['name']
    code = delete(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}',
                  TOKEN, f'catalog: {cname}')
    if code not in (200, 204):
        # show what's still blocking it
        nss = namespaces_deep_first(cname, TOKEN)
        crs = [r['name'] for r in requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN)).json().get('roles', []) if r['name'] != 'catalog_admin']
        print(f'      ↳ blocked — remaining namespaces: {[ns_label(n) for n in nss]}, non-default roles: {crs}')

print('\n✅ Step 4 complete')


=== Step 4: Delete Catalogs ===
  ✅ [204] catalog: user10_catalog
  ✅ [204] catalog: user16_catalog
  ✅ [204] catalog: user19_catalog
  ✅ [204] catalog: user1_catalog
  ✅ [204] catalog: user20_catalog
  ✅ [204] catalog: user22_catalog
  ✅ [204] catalog: user25_catalog
  ✅ [204] catalog: user26_catalog
  ✅ [204] catalog: user28_catalog
  ✅ [204] catalog: user29_catalog
  ✅ [204] catalog: user2_catalog
  ✅ [204] catalog: user30_catalog
  ✅ [204] catalog: user31_catalog
  ✅ [204] catalog: user32_catalog
  ✅ [204] catalog: user33_catalog
  ✅ [204] catalog: user34_catalog
  ✅ [204] catalog: user36_catalog
  ✅ [204] catalog: user37_catalog
  ✅ [204] catalog: user38_catalog
  ✅ [204] catalog: user39_catalog
  ✅ [204] catalog: user3_catalog
  ✅ [204] catalog: user40_catalog
  ✅ [204] catalog: user41_catalog
  ✅ [204] catalog: user42_catalog
  ✅ [204] catalog: user43_catalog
  ✅ [204] catalog: user45_catalog
  ✅ [204] catalog: user46_catalog
  ✅ [204] catalog: user49_catalog
  ✅ [204] catalog: 

In [42]:
# ============================================================
# Step 5 — Delete Principals
# ============================================================
print('=== Step 5: Delete Principals ===')

for p in principals:
    pname = p['name']
    if pname in SKIP_PRINCIPALS:
        print(f'  ⏭️  skip: {pname}')
        continue
    delete(
        f'{POLARIS_URL}/api/management/v1/principals/{pname}',
        TOKEN, f'principal: {pname}'
    )

print('\n✅ Step 5 complete')

=== Step 5: Delete Principals ===
  ⏭️  skip: root
  ✅ [204] principal: user10_principal
  ✅ [204] principal: user16_principal
  ✅ [204] principal: user19_principal
  ✅ [204] principal: user1_principal
  ✅ [204] principal: user20_principal
  ✅ [204] principal: user22_principal
  ✅ [204] principal: user25_principal
  ✅ [204] principal: user26_principal
  ✅ [204] principal: user28_principal
  ✅ [204] principal: user29_principal
  ✅ [204] principal: user2_principal
  ✅ [204] principal: user30_principal
  ✅ [204] principal: user31_principal
  ✅ [204] principal: user32_principal
  ✅ [204] principal: user33_principal
  ✅ [204] principal: user34_principal
  ✅ [204] principal: user36_principal
  ✅ [204] principal: user37_principal
  ✅ [204] principal: user38_principal
  ✅ [204] principal: user39_principal
  ✅ [204] principal: user3_principal
  ✅ [204] principal: user40_principal
  ✅ [204] principal: user41_principal
  ✅ [204] principal: user42_principal
  ✅ [204] principal: user43_principal
  

In [43]:
# ============================================================
# Step 6 — Delete Principal Roles
# ============================================================
print('=== Step 6: Delete Principal Roles ===')

for pr in principal_roles:
    prname = pr['name']
    if prname in SKIP_PRINCIPAL_ROLES:
        print(f'  ⏭️  skip: {prname}')
        continue
    delete(
        f'{POLARIS_URL}/api/management/v1/principal-roles/{prname}',
        TOKEN, f'principal-role: {prname}'
    )

print('\n✅ Step 6 complete')

=== Step 6: Delete Principal Roles ===
  ⏭️  skip: service_admin
  ✅ [204] principal-role: user10_principal_role
  ✅ [204] principal-role: user16_principal_role
  ✅ [204] principal-role: user19_principal_role
  ✅ [204] principal-role: user1_principal_role
  ✅ [204] principal-role: user20_principal_role
  ✅ [204] principal-role: user22_principal_role
  ✅ [204] principal-role: user25_principal_role
  ✅ [204] principal-role: user26_principal_role
  ✅ [204] principal-role: user28_principal_role
  ✅ [204] principal-role: user29_principal_role
  ✅ [204] principal-role: user2_principal_role
  ✅ [204] principal-role: user30_principal_role
  ✅ [204] principal-role: user31_principal_role
  ✅ [204] principal-role: user32_principal_role
  ✅ [204] principal-role: user33_principal_role
  ✅ [204] principal-role: user34_principal_role
  ✅ [204] principal-role: user36_principal_role
  ✅ [204] principal-role: user37_principal_role
  ✅ [204] principal-role: user38_principal_role
  ✅ [204] principal-role:

In [44]:
# ============================================================
# Verify — confirm everything is clean
# ============================================================
print('=== Verification ===')

remaining = {
    'catalogs':       requests.get(f'{POLARIS_URL}/api/management/v1/catalogs',        headers=h(TOKEN)).json().get('catalogs', []),
    'principals':     requests.get(f'{POLARIS_URL}/api/management/v1/principals',      headers=h(TOKEN)).json().get('principals', []),
    'principal_roles': requests.get(f'{POLARIS_URL}/api/management/v1/principal-roles', headers=h(TOKEN)).json().get('roles', []),
}

all_clean = True
for resource, items in remaining.items():
    names = [i['name'] for i in items]
    icon = '✅' if len(names) == 0 or all(n in SKIP_PRINCIPALS | SKIP_PRINCIPAL_ROLES for n in names) else '⚠️ '
    if icon == '⚠️ ':
        all_clean = False
    print(f'  {icon} {resource}: {names}')

print()
if all_clean:
    print('✅ All resources cleaned — only system entities remain')
else:
    print('⚠️  Some resources remain — check above')

=== Verification ===
  ✅ catalogs: []
  ✅ principals: ['root']
  ✅ principal_roles: ['service_admin']

✅ All resources cleaned — only system entities remain
